# AdaLux: Training

Allena PPO e SAC su `AdaLux` e salva modelli (`models/`) e log di
training (`logs/`) su disco. L'analisi delle tre domande di ricerca
(Q1/Q2/Q3) è nel notebook companion `02_domande_di_ricerca.ipynb`, che
carica questi modelli già allenati senza bisogno di rieseguire il
training.

## 0. Setup

In [1]:
import os
import sys
import time

import numpy as np
import torch
from tqdm.auto import tqdm

from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import BaseCallback

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "modules")))
from eval_utils import make_monitored_env


class TqdmProgressCallback(BaseCallback):
    def __init__(self, total_timesteps, desc="Training"):
        super().__init__()
        self.total_timesteps = total_timesteps
        self.desc = desc
        self.pbar = None

    def _on_training_start(self):
        remaining = self.total_timesteps - self.model.num_timesteps
        self.pbar = tqdm(total=max(remaining, 0), desc=self.desc, unit="step")

    def _on_step(self):
        self.pbar.update(self.training_env.num_envs)
        return True

    def _on_training_end(self):
        self.pbar.n = self.pbar.total
        self.pbar.refresh()
        self.pbar.close()

RNG_SEED = 42
np.random.seed(RNG_SEED)

MODELS_DIR = "../models"
LOGS_DIR = "../logs"
for d in (MODELS_DIR, LOGS_DIR):
    os.makedirs(d, exist_ok=True)

In [2]:
# modalità rapida per testing
FAST_MODE = False
# Numero di training seed per PPO e SAC
N_SEEDS = 3
TIMESTEPS = 20_000 if FAST_MODE else 80_000
N_TEST_EPISODES = 10 if FAST_MODE else 30  # non usato qui, ma deve combaciare
                                           # con il valore in 02_domande_di_ricerca.ipynb

print(f"FAST_MODE={FAST_MODE} | N_SEEDS={N_SEEDS} | TIMESTEPS={TIMESTEPS} - "
      f"N_TEST_EPISODES={N_TEST_EPISODES}")


FAST_MODE=False | N_SEEDS=3 | TIMESTEPS=80000 - N_TEST_EPISODES=30


In [3]:
DEVICE = "cpu" # è consigliabile usare cpu per via del costo del passaggio dei tensori fra GPU e CPU

cuda_available = torch.cuda.is_available()
effective_device = ("cuda" if cuda_available else "cpu") if DEVICE == "auto" else DEVICE
print(f"DEVICE = {DEVICE!r}")


DEVICE = 'cpu'


## 1. Training PPO e SAC (multi-seed)

Alleniamo **PPO** e **SAC** sullo stesso ambiente, per lo stesso numero di
timestep (`TIMESTEPS`) e la stessa architettura di rete (`[128, 128]`),
ripetuto su **`N_SEEDS` training seed indipendenti**.

Ogni modello viene salvato singolarmente (`models/ppo_q2_seed{n}.zip`,
`models/sac_q2_seed{n}.zip`) e ricaricato da disco se già presente
(`RETRAIN = False`, default): non serve riallenare tutto da zero ogni
volta che riapri il notebook, e un'interruzione a metà non fa perdere il
lavoro già fatto.

In [4]:
def train_and_log(algo_name, timesteps, log_dir, seed=RNG_SEED, device=DEVICE,
                   save_path=None, retrain=True):
    #Allena PPO/SAC loggando il reward per episodio con Monitor in log_dir.
    #Se save_path è indicato, il modello addestrato viene salvato su disco
    #(save_path + ".zip"). Se retrain=False e in save_path esiste già un
    #modello salvato, lo carica invece di riallenare da zero: utile per non
    #dover riallenare da zero ogni volta che riapri il notebook.
    algo_cls = {"PPO": PPO, "SAC": SAC}.get(algo_name)
    if algo_cls is None:
        raise ValueError(algo_name)

    if not retrain and save_path is not None and os.path.exists(save_path + ".zip"):
        print(f"Carico {algo_name} già addestrato da {save_path}.zip "
              f"(retrain=False: le curve di reward restano quelle del training precedente)")
        return algo_cls.load(save_path, device=device)

    env = make_monitored_env(seed=seed, log_dir=log_dir)
    common_kwargs = dict(verbose=0, seed=seed, device=device, policy_kwargs=dict(net_arch=[128, 128]))
    if algo_name == "PPO":
        model = PPO("MlpPolicy", env, n_steps=256, batch_size=256, n_epochs=10,
                    gamma=0.995, gae_lambda=0.95, learning_rate=3e-4,
                    ent_coef=0.01, clip_range=0.2, **common_kwargs)
    else:  # SAC
        model = SAC("MlpPolicy", env, learning_rate=3e-4, buffer_size=200_000,
                    batch_size=256, gamma=0.995, tau=0.005, train_freq=1,
                    gradient_steps=1, ent_coef="auto", **common_kwargs)

    t0 = time.time()
    model.learn(total_timesteps=timesteps, log_interval=50,
                callback=TqdmProgressCallback(timesteps, desc=algo_name))
    print(f"{algo_name}: training di {timesteps} step completato in {time.time() - t0:.1f}s")
    if save_path is not None:
        model.save(save_path)
        print(f"Modello salvato in {save_path}.zip")
    return model


SEEDS = list(range(1, N_SEEDS + 1))
RETRAIN = False  # True per riallenare da zero anche se i modelli sono gia' stati salvati

models = {}
logs = {}
for seed in tqdm(SEEDS, desc="Training PPO+SAC per seed"):
    for algo_name in ("PPO", "SAC"):
        log_dir = os.path.join(LOGS_DIR, f"q2_{algo_name.lower()}_seed{seed}")
        save_path = os.path.join(MODELS_DIR, f"{algo_name.lower()}_q2_seed{seed}")
        model = train_and_log(algo_name, TIMESTEPS, log_dir, seed=seed,
                               save_path=save_path, retrain=RETRAIN)
        models[(algo_name, seed)] = model
        logs[(algo_name, seed)] = log_dir

print(f"{len(models)} modelli pronti ({len(SEEDS)} seed x 2 algoritmi).")


Training PPO+SAC per seed:   0%|          | 0/3 [00:00<?, ?it/s]

Carico PPO già addestrato da ../models\ppo_q2_seed1.zip (retrain=False : le curve di reward restano quelle del training precedente)
Carico SAC già addestrato da ../models\sac_q2_seed1.zip (retrain=False : le curve di reward restano quelle del training precedente)
Carico PPO già addestrato da ../models\ppo_q2_seed2.zip (retrain=False : le curve di reward restano quelle del training precedente)
Carico SAC già addestrato da ../models\sac_q2_seed2.zip (retrain=False : le curve di reward restano quelle del training precedente)
Carico PPO già addestrato da ../models\ppo_q2_seed3.zip (retrain=False : le curve di reward restano quelle del training precedente)
Carico SAC già addestrato da ../models\sac_q2_seed3.zip (retrain=False : le curve di reward restano quelle del training precedente)
6 modelli pronti (3 seed x 2 algoritmi).
